# fig-vip-ephys notebook

Reproduces panels A–D of `fig-vip-ephys.png`. Quantitative entries are taken verbatim from the Phase-6-audited `figure_data` of `cluster_04_intrinsic_electrophysiology` (see `section_05_figure_assignments.json`).

**Caveats embedded in caption (verbatim from Phase-6 audit):**

- Panel B (R$_{in}$): Cross-region (CA1 stratum radiatum vs BLA) — must be flagged in any figure caption. | n=2 entries — minimal cross-paper comparison.
- Panel C (IS-fraction): Denominators differ across entries; per-entry denominators MUST be shown in the figure (cannot be presented as a single comparable percentage without that annotation). | Rat (Cauli) vs mouse (Anastasiades) species mismatch. | Different cortical areas (S1, frontal cortex, mPFC L1b) and slightly different firing-class taxonomies.
- Panel D (τ$_m$): single-row entry after iter3 audit removed Rhomberg 2018 (τ$_m$ value not verifiable from packet).

In [ ]:
import sys; sys.path.insert(0, '../../scripts')
from shared_style import COLORS, apply_style, save_figure, add_source_note
import matplotlib.pyplot as plt
import numpy as np
apply_style()

In [ ]:
# Cross-study entries (from cluster_04_intrinsic_electrophysiology figure_data)
rin_entries = [
    {'cite': 'Michaud 2024',  'label': 'Michaud 2024\n(CA1, VIP/CR+ I-S3, n=7)',   'value': 517,   'sem': 37,    'color': COLORS['blue']},
    {'cite': 'Rhomberg 2018', 'label': 'Rhomberg 2018\n(BLA, VIP+ IS-INs, n=30)',  'value': 350.3, 'sem': 28.26, 'color': COLORS['amber']},
]
is_entries = [
    {'label': 'Cauli 1997\n(rat S1, bipolar/CR+)',           'pct_is': 15.5, 'denom': '15/97'},
    {'label': 'Cauli 2000\n(rat frontal, fusiform clust.)',  'pct_is': 26.7, 'denom': '16/60'},
    {'label': 'Pronneke 2015\n(mouse barrel, VIP-Cre)',      'pct_is': 14.5, 'denom': '~14.5%'},
    {'label': 'Anastasiades 2021\n(mouse mPFC L1b, VIP-Cre)','pct_is': 29.4, 'denom': '5/17'},
]
tau_entry = {'label': 'Michaud 2024\n(CA1 strat. radiatum,\nVIP/CR+ I-S3, n=7)', 'value': 36.5, 'sem': 3.2, 'color': COLORS['blue']}

In [ ]:
def gen_AP(t, spike_times):
    v = -65 * np.ones_like(t)
    for st in spike_times:
        idx = (t > st-0.002) & (t < st+0.005)
        local = t[idx] - st
        v[idx] = np.maximum(v[idx], -65 + 105*np.exp(-((local)/0.0008)**2))
    return v

fig = plt.figure(figsize=(13, 11))
gs = fig.add_gridspec(2, 2, hspace=0.55, wspace=0.32)

# Panel A — exemplar firing traces
axA = fig.add_subplot(gs[0, 0])
t = np.linspace(0, 1.0, 2000)
is_spikes = sorted([0.10, 0.12, 0.13, 0.30, 0.31, 0.55, 0.56, 0.58, 0.83, 0.85])
v_is = gen_AP(t, is_spikes)
ca_isi = np.array([0.07, 0.085, 0.105, 0.13, 0.16, 0.20, 0.25, 0.31, 0.37])
ca_spikes = np.cumsum(ca_isi) + 0.10; ca_spikes = ca_spikes[ca_spikes < 0.95]
v_ca = gen_AP(t, list(ca_spikes))
acc_isi = np.array([0.06, 0.075, 0.10, 0.15, 0.22, 0.32])
acc_spikes = np.cumsum(acc_isi) + 0.10; acc_spikes = acc_spikes[acc_spikes < 0.95]
v_acc = gen_AP(t, list(acc_spikes))
off=90
axA.plot(t, v_is + 2*off, color=COLORS['red'], linewidth=1.2)
axA.plot(t, v_ca + off,   color=COLORS['blue'], linewidth=1.2)
axA.plot(t, v_acc,        color=COLORS['green'], linewidth=1.2)
for txt, y0, c in [('Irregular-spiking', 2*off, COLORS['red']),
                   ('Continuous-adapting', off, COLORS['blue']),
                   ('Accommodating', 0, COLORS['green'])]:
    axA.text(1.02, -65 + y0, txt, color=c, fontsize=10, va='center')
axA.plot([0.05, 0.95], [-180, -180], color=COLORS['gray_700'], lw=2)
axA.text(0.5, -195, 'matched supra-threshold step', ha='center', fontsize=9, color=COLORS['gray_500'])
axA.plot([0.85, 0.95], [-150, -150], color='black', lw=1.5)
axA.plot([0.85, 0.85], [-150, -100], color='black', lw=1.5)
axA.text(0.90, -158, '100 ms', ha='center', fontsize=8)
axA.text(0.83, -125, '50 mV', ha='right', fontsize=8, rotation=90, va='center')
axA.set_xlim(0, 1.20); axA.set_ylim(-220, 200); axA.axis('off')
axA.set_title('A  Exemplar firing patterns (cortical VIP)', loc='left', fontsize=12, fontweight='bold')

In [ ]:
# Panels B, C, D — see main workspace builder for full code; this notebook reproduces
# the same figure using the same shared_style helpers.
axB = fig.add_subplot(gs[0, 1])
ypos = np.arange(len(rin_entries))[::-1]
for i, e in enumerate(rin_entries):
    axB.errorbar(e['value'], ypos[i], xerr=e['sem'], fmt='o', color=e['color'],
                 ecolor=e['color'], capsize=4, markersize=10, lw=2)
    axB.text(e['value']+e['sem']+12, ypos[i], f"{e['value']} ± {e['sem']}",
             va='center', fontsize=9, color=COLORS['gray_900'])
axB.set_yticks(ypos); axB.set_yticklabels([e['label'] for e in rin_entries], fontsize=9)
axB.set_xlabel('Input resistance R$_{in}$ (MΩ)')
axB.set_xlim(150, 700); axB.set_ylim(-0.6, 1.6); axB.grid(axis='x', alpha=0.3)
axB.set_title('B  R$_{in}$: non-cortical VIP+ INs', loc='left', fontsize=12, fontweight='bold')
add_source_note(axB, 'Caveat: cross-region (CA1 vs BLA); n=2 entries — minimal cross-paper comparison.', y=-0.30)

In [ ]:
axC = fig.add_subplot(gs[1, 0])
ypos = np.arange(len(is_entries))[::-1]
for i, e in enumerate(is_entries):
    y = ypos[i]
    axC.barh(y, e['pct_is'], color=COLORS['red'], alpha=0.85,
             label='IS' if i==0 else None, edgecolor='white')
    axC.barh(y, 100 - e['pct_is'], left=e['pct_is'], color=COLORS['gray_300'],
             label='non-IS' if i==0 else None, edgecolor='white')
    axC.text(e['pct_is']/2, y, f"{e['pct_is']:.1f}%", ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    axC.text(101, y, f"denom: {e['denom']}", va='center', fontsize=8, color=COLORS['gray_700'])
axC.set_yticks(ypos); axC.set_yticklabels([e['label'] for e in is_entries], fontsize=9)
axC.set_xlabel('% of VIP+ cells with IS firing')
axC.set_xlim(0, 130); axC.set_ylim(-0.6, len(is_entries)-0.4)
axC.legend(loc='upper right', fontsize=8, frameon=False, bbox_to_anchor=(1.0, -0.06), ncol=2)
axC.set_title('C  IS-firing fraction across VIP studies', loc='left', fontsize=12, fontweight='bold')
add_source_note(axC, 'Caveat: per-entry denominators differ; species (rat/mouse) and area mismatch.', y=-0.38)

# Panel D — single retained entry
axD = fig.add_subplot(gs[1, 1])
e = tau_entry
axD.errorbar(e['value'], 0.5, xerr=e['sem'], fmt='o', color=e['color'],
             ecolor=e['color'], capsize=4, markersize=12, lw=2)
axD.text(e['value']+e['sem']+1.5, 0.5, f"{e['value']} ± {e['sem']} ms",
         va='center', fontsize=10, color=COLORS['gray_900'])
axD.text(0.5, 0.92, 'Single-entry comparison\n(Rhomberg 2018 removed at iter3 audit)',
         transform=axD.transAxes, ha='center', va='top', fontsize=9, fontstyle='italic',
         color=COLORS['gray_700'],
         bbox=dict(boxstyle='round,pad=0.4', fc=COLORS['gray_100'], ec=COLORS['gray_300']))
axD.set_yticks([0.5]); axD.set_yticklabels([e['label']], fontsize=9)
axD.set_xlabel('Membrane time constant τ$_m$ (ms)')
axD.set_xlim(20, 55); axD.set_ylim(0, 1.1); axD.grid(axis='x', alpha=0.3)
axD.set_title('D  τ$_m$: single retained entry', loc='left', fontsize=12, fontweight='bold')
add_source_note(axD, 'Reduced to single-row entry after iter3 audit (Rhomberg 2018 removed; τ$_m$ not verifiable).', y=-0.30)

fig.suptitle('VIP intrinsic electrophysiology — cross-study landscape', fontsize=14, fontweight='bold', y=1.00)
save_figure(fig, '../fig-vip-ephys.png')